# IndicWav2Vec — Kathbath (Vistaar) Test-Set Evaluation — Bengali

Evaluates `ai4bharat/indicwav2vec_v1_bengali` on the **Kathbath** benchmark test set
(via the official Vistaar release) for **Bengali**.

**No manual dataset upload needed** - the notebook downloads the official combined
benchmark zip directly from AI4Bharat's object store and extracts only the language you
need. The Bengali folder has a `manifest.json` (NeMo-style JSONL: one
`{"audio_filepath", "duration", "text"}` object per line) and a `wavs/` folder of audio.

**Note on IndicWav2Vec vs IndicConformer:** unlike IndicConformer (one multilingual
checkpoint with a `language_code` argument and CTC/RNNT decoding modes), IndicWav2Vec
ships one fine-tuned checkpoint per language, loaded through the standard HuggingFace
`automatic-speech-recognition` pipeline. There's no CTC/RNNT split here - just a single
greedy CTC decode per utterance - so the summary table has one WER/CER column instead of
two decoding passes.

**Before running:**
1. (Optional) Set an HF token in the login cell - `ai4bharat/indicwav2vec_v1_bengali` is
   a public model, so this is only needed if you're hitting HF rate limits.
2. Run cells top to bottom - the download/extract cell handles Bengali automatically.


In [14]:
# This Python 3 environment comes with many helpful analytics libraries installed
import numpy as np
import pandas as pd
import os

for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames[:5]:
        print(os.path.join(dirname, filename))


In [15]:
!nvidia-smi


Tue Jul 21 11:17:37 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.159.04             Driver Version: 580.159.04     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   54C    P0             27W /   70W |     167MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [16]:
!pip install -q transformers torchaudio "onnxruntime==1.20.1" "onnx==1.20.1"
!pip install -q jiwer


## Download the Kathbath (Vistaar) benchmark automatically

No manual upload needed. AI4Bharat hosts the Vistaar benchmark test sets as a single combined
zip covering all 12 languages: `kathbath.zip` from
https://github.com/AI4Bharat/vistaar (see "Download Training Datasets and Benchmarks").

The zip's internal layout is `kathbath/<lang>/manifest.json` + `kathbath/<lang>/wavs/*.wav`,
so no manifest editing is needed here either.

Since it's one zip for all languages, the cell below downloads the whole archive once
(cached - re-running skips re-download), then **selectively extracts only the languages
listed in `LANGS_TO_EXTRACT`** so you're not paying disk/time for languages you're not
using yet. Starting with Bengali only.


In [17]:
import os

ZIP_URL = "https://indicwhisper.objectstore.e2enetworks.net/vistaar_benchmarks/kathbath.zip"
DOWNLOAD_DIR = "/kaggle/working/kathbath_download"
ZIP_PATH = os.path.join(DOWNLOAD_DIR, "kathbath.zip")

os.makedirs(DOWNLOAD_DIR, exist_ok=True)


def zip_is_valid(path):
    """Returns True only if the file exists AND is a complete, openable zip archive.
    A partial/interrupted download can still have a valid-looking header, so we
    actually try to open it and read the central directory."""
    if not os.path.exists(path) or os.path.getsize(path) == 0:
        return False
    try:
        import zipfile
        with zipfile.ZipFile(path, "r") as zf:
            bad_file = zf.testzip()  # returns None if all entries check out
            return bad_file is None
    except Exception:
        return False


if zip_is_valid(ZIP_PATH):
    print(f"Zip already downloaded and verified at {ZIP_PATH}, skipping download.")
else:
    if os.path.exists(ZIP_PATH):
        print(f"Found an existing file at {ZIP_PATH} but it's incomplete/corrupt "
              f"(size={os.path.getsize(ZIP_PATH) / 1e9:.2f} GB) - resuming download.")
    else:
        print("Downloading Kathbath (Vistaar benchmark) zip - single archive covering all 12")
        print("languages, so this can take a while depending on connection speed...")
    # -c resumes from where a partial file left off instead of restarting from zero
    !wget -c -O {ZIP_PATH} {ZIP_URL}

    if not zip_is_valid(ZIP_PATH):
        raise RuntimeError(
            f"Download finished but {ZIP_PATH} is still not a valid/complete zip "
            f"(size={os.path.getsize(ZIP_PATH) / 1e9:.2f} GB). This usually means the "
            f"session was interrupted again, or the server doesn't support resumable "
            f"range requests for this URL. Try re-running this cell (wget -c will keep "
            f"resuming), or delete {ZIP_PATH} to force a full fresh download."
        )
    print("Zip verified OK.")

print(f"Zip size: {os.path.getsize(ZIP_PATH) / 1e9:.2f} GB")


Zip already downloaded and verified at /kaggle/working/kathbath_download/kathbath.zip, skipping download.
Zip size: 3.46 GB


In [18]:
import zipfile
from tqdm import tqdm

EXTRACT_ROOT = "/kaggle/working/kathbath_data"
os.makedirs(EXTRACT_ROOT, exist_ok=True)

# Languages to extract right now. Add more later ("hindi", "tamil", "marathi", "telugu")
# and re-run this cell to bring in additional languages without re-downloading.
LANGS_TO_EXTRACT = ["bengali"]

with zipfile.ZipFile(ZIP_PATH, "r") as zf:
    all_names = zf.namelist()
    top_level = all_names[0].split("/")[0]
    print(f"Top-level folder inside zip: '{top_level}'")

    already_extracted = {
        lang for lang in LANGS_TO_EXTRACT
        if os.path.exists(os.path.join(EXTRACT_ROOT, top_level, lang, "manifest.json"))
    }
    to_process = [lang for lang in LANGS_TO_EXTRACT if lang not in already_extracted]
    if already_extracted:
        print(f"Already extracted, skipping: {sorted(already_extracted)}")
    if not to_process:
        print("Nothing new to extract.")
    else:
        to_extract = [
            n for n in all_names
            if any(n.startswith(f"{top_level}/{lang}/") for lang in to_process)
        ]
        print(f"Extracting {len(to_extract)} files for: {to_process} ...")
        for name in tqdm(to_extract):
            zf.extract(name, EXTRACT_ROOT)

KATHBATH_ROOT = os.path.join(EXTRACT_ROOT, top_level)
print(f"\nKATHBATH_ROOT = {KATHBATH_ROOT}")
print(f"Contents: {os.listdir(KATHBATH_ROOT)}")


Top-level folder inside zip: 'kathbath'
Extracting 1786 files for: ['bengali'] ...



100%|██████████| 1786/1786 [00:02<00:00, 615.93it/s]


KATHBATH_ROOT = /kaggle/working/kathbath_data/kathbath
Contents: ['bengali', 'hindi']


In [19]:
from huggingface_hub import login
import os

# SECURITY: don't hardcode HF tokens in the notebook. Use Kaggle's built-in
# "Add-ons > Secrets" (or an environment variable) so the token isn't stored in
# plain text in the .ipynb file, which is easy to accidentally share/commit.
# Optional for IndicWav2Vec - the model is public - but useful if you hit rate limits.
hf_token = os.environ.get("HF_TOKEN")
if not hf_token:
    try:
        from kaggle_secrets import UserSecretsClient
        hf_token = UserSecretsClient().get_secret("HF_TOKEN")
    except Exception:
        hf_token = None

if hf_token:
    login(token=hf_token)
    print("Successfully logged into Hugging Face Hub!")
else:
    print("No HF token found - continuing without login (fine for this public model).")


Successfully logged into Hugging Face Hub!


In [20]:
from transformers import pipeline
import torch

# 1. Load the Bengali IndicWav2Vec model
print("Loading the model... (this will take a few minutes on first run)")
device_index = 0 if torch.cuda.is_available() else -1
pipe = pipeline(
    "automatic-speech-recognition",
    model="ai4bharat/indicwav2vec_v1_bengali",
    device=device_index,
)

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Model loaded on {device}")


Loading the model... (this will take a few minutes on first run)


config.json: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/1.26G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/424 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/257 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.26G [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/940 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/85.0 [00:00<?, ?B/s]

preprocessor_config.json:   0%|          | 0.00/254 [00:00<?, ?B/s]

Model loaded on cuda


## Config

Point `KATHBATH_ROOT` at wherever your attached Kaggle Dataset lands - check the file
listing printed in the first cell, or the "Data" panel on the right, to confirm the exact
path (it's usually `/kaggle/input/<your-dataset-slug>`, which may or may not itself
contain a `kathbath/` subfolder depending on how you zipped it).

`KATHBATH_ROOT/bengali/` is expected to contain:
- `manifest.json` - one JSON object per line: `{"audio_filepath": ..., "duration": ..., "text": ...}`
- an audio subfolder (commonly `wav/` or `wavs/`) - the exact name doesn't matter, the
  notebook indexes every `.wav`/`.flac`/`.mp3` file under the language folder by filename.

Sample count reported (Vistaar-filtered Kathbath test set):

| Language | Folder    | HF model                              | Test samples |
|---|---|---|---|
| Bengali  | `bengali` | `ai4bharat/indicwav2vec_v1_bengali`   | 1,783 |

`IndicWav2Vec` ships one checkpoint per language (no shared multilingual model, no
`language_code` argument), so `MODEL_NAME` and `LANGUAGES` below only ever hold a single
entry - swap the model repo id and folder name to point this notebook at another language.


In [21]:
# ── KATHBATH_ROOT is already set by the download/extract cell above.
# If you're attaching a manually-uploaded Kaggle Dataset instead, uncomment and edit:
# KATHBATH_ROOT = "/kaggle/input/kathbath"

# (kathbath_folder_name, display_name) - only languages actually extracted above will
# work. IndicWav2Vec has one checkpoint per language, so add an entry here (and load a
# new `pipe` above pointed at the matching ai4bharat/indicwav2vec_v1_<lang> repo) if you
# want to extend this to more languages.
LANGUAGES = [
    ("bengali", "Bengali"),
]

# Set to an int (e.g. 300) to cap samples for a quick test run, or None to evaluate
# every sample in the manifest.json.
SAMPLES_PER_LANG = None

CHECKPOINT_EVERY = 100
CHECKPOINT_DIR = "/kaggle/working"


In [22]:
import json
import os
from pathlib import Path
import torch
import torchaudio
import numpy as np
from tqdm import tqdm
from jiwer import wer, cer


def read_manifest(manifest_path):
    """Reads a NeMo-style manifest: one JSON object per line. Falls back to a single
    JSON array if the file isn't line-delimited."""
    with open(manifest_path, "r", encoding="utf-8") as f:
        content = f.read().strip()
    entries = []
    try:
        for line in content.splitlines():
            line = line.strip()
            if not line:
                continue
            entries.append(json.loads(line))
    except json.JSONDecodeError:
        entries = json.loads(content)
    return entries


def build_audio_index(lang_dir: Path) -> dict:
    """Scans lang_dir recursively once and maps basename -> full path. Robust to any
    audio subfolder naming ('wav', 'wavs', etc.) and any nesting depth."""
    index = {}
    for ext in ("*.wav", "*.flac", "*.mp3"):
        for p in lang_dir.rglob(ext):
            index[p.name] = str(p)
    return index


def resolve_audio_path(lang_dir: Path, audio_filepath: str, audio_index: dict) -> str:
    """audio_filepath in the manifest may be absolute (from the original download
    machine), relative to some root above lang_dir (e.g. 'kathbath/bengali/wavs/x.wav'),
    or just a bare filename. Try direct candidates first, then fall back to the
    prebuilt basename index (handles any subfolder naming mismatch, e.g. 'wav' vs 'wavs')."""
    p = Path(audio_filepath)
    candidates = [
        p if p.is_absolute() else None,
        lang_dir / audio_filepath,
        lang_dir.parent / audio_filepath,  # manifest path already includes '<lang>/wavs/...'
        lang_dir / "wav" / p.name,
        lang_dir / "wavs" / p.name,
        lang_dir / p.name,
    ]
    for c in candidates:
        if c is not None and c.exists():
            return str(c)
    if p.name in audio_index:
        return audio_index[p.name]
    raise FileNotFoundError(
        f"Could not resolve audio file '{audio_filepath}' under {lang_dir} "
        f"(tried direct path candidates and a full basename index of {len(audio_index)} "
        f"audio files under this folder)."
    )


def checkpoint_path(lang_folder):
    return os.path.join(CHECKPOINT_DIR, f"checkpoint_{lang_folder}.json")


def save_checkpoint(idx, refs, preds, lang_folder):
    path = checkpoint_path(lang_folder)
    data = {
        "last_index": idx,
        "references": refs,
        "predictions": preds,
    }
    tmp = path + ".tmp"
    with open(tmp, "w", encoding="utf-8") as f:
        json.dump(data, f, ensure_ascii=False, indent=2)
    os.replace(tmp, path)
    print(f"  \u2714 Checkpoint saved at sample {idx} \u2192 {path}")


def load_checkpoint(lang_folder):
    path = checkpoint_path(lang_folder)
    if os.path.exists(path):
        with open(path, "r", encoding="utf-8") as f:
            data = json.load(f)
        print(f"Resuming {lang_folder} from checkpoint: {data['last_index'] + 1} samples already done.")
        return data["last_index"] + 1, data["references"], data["predictions"]
    return 0, [], []


def load_audio_array(audio_source, target_sr=16000):
    """Loads a wav file, downmixes to mono, resamples to target_sr, and returns a 1-D
    numpy float32 array (what the HF ASR pipeline expects via {"raw": ..., "sampling_rate": ...})."""
    waveform, original_sr = torchaudio.load(audio_source)
    if waveform.shape[0] > 1:
        waveform = torch.mean(waveform, dim=0, keepdim=True)
    if original_sr != target_sr:
        resampler = torchaudio.transforms.Resample(orig_freq=original_sr, new_freq=target_sr)
        waveform = resampler(waveform)
    return waveform.squeeze(0).numpy(), target_sr


def evaluate_language(lang_folder, display_name, kathbath_root, n_samples=None):
    """Reads manifest.json for one language, runs IndicWav2Vec inference over its audio
    via the HF ASR pipeline, checkpointing periodically. Returns (references, predictions)."""
    print(f"\n{'='*70}\nEvaluating: {display_name}  (folder={lang_folder})\n{'='*70}")

    lang_dir = Path(kathbath_root) / lang_folder
    manifest_path = lang_dir / "manifest.json"
    if not manifest_path.exists():
        raise FileNotFoundError(f"manifest.json not found at {manifest_path} - check KATHBATH_ROOT.")

    entries = read_manifest(manifest_path)
    if n_samples is not None:
        entries = entries[:n_samples]
    total = len(entries)
    print(f"Found {total} entries in manifest.")

    print("Indexing audio files...")
    audio_index = build_audio_index(lang_dir)
    print(f"Indexed {len(audio_index)} audio files under {lang_dir}.")

    start_idx, references, predictions = load_checkpoint(lang_folder)

    last_idx = start_idx - 1
    for sample_idx in tqdm(range(start_idx, total), initial=start_idx, total=total, desc=display_name):
        entry = entries[sample_idx]
        reference_text = entry["text"]

        try:
            wav_path = resolve_audio_path(lang_dir, entry["audio_filepath"], audio_index)
            audio_array, sr = load_audio_array(wav_path)

            transcription = pipe({"sampling_rate": sr, "raw": audio_array})["text"]

            references.append(reference_text)
            predictions.append(transcription.strip())
            last_idx = sample_idx

        except (RuntimeError, FileNotFoundError) as e:
            print(f"Skipping sample {sample_idx} due to error: {e}.")
            continue

        if (sample_idx + 1) % CHECKPOINT_EVERY == 0:
            save_checkpoint(sample_idx, references, predictions, lang_folder)

    save_checkpoint(last_idx, references, predictions, lang_folder)
    print(f"Inference for {display_name} complete: {len(references)} samples evaluated.")
    return references, predictions


In [23]:
# ── Run evaluation for Bengali ────────────────────────────────────────────────
results = {}

for lang_folder, display_name in LANGUAGES:
    refs, preds = evaluate_language(
        lang_folder, display_name, KATHBATH_ROOT, n_samples=SAMPLES_PER_LANG
    )
    results[display_name] = {
        "references": refs,
        "predictions": preds,
    }



Evaluating: Bengali  (folder=bengali)
Found 1783 entries in manifest.
Indexing audio files...
Indexed 1783 audio files under /kaggle/working/kathbath_data/kathbath/bengali.



Bengali:   1%|          | 10/1783 [00:02<04:26,  6.64it/s]You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset

Bengali:   6%|▌         | 101/1783 [00:12<03:01,  9.27it/s]

  ✔ Checkpoint saved at sample 99 → /kaggle/working/checkpoint_bengali.json



Bengali:  11%|█▏        | 201/1783 [00:22<02:32, 10.35it/s]

  ✔ Checkpoint saved at sample 199 → /kaggle/working/checkpoint_bengali.json


...

  ✔ Checkpoint saved at sample 1599 → /kaggle/working/checkpoint_bengali.json



Bengali:  95%|█████████▌| 1701/1783 [02:58<00:09,  9.00it/s]

  ✔ Checkpoint saved at sample 1699 → /kaggle/working/checkpoint_bengali.json



Bengali: 100%|██████████| 1783/1783 [03:07<00:00,  9.50it/s]

  ✔ Checkpoint saved at sample 1782 → /kaggle/working/checkpoint_bengali.json
Inference for Bengali complete: 1783 samples evaluated.


In [24]:
# ── Compute WER/CER per language and build a summary table ──────────────────
rows = []
for display_name, r in results.items():
    refs, preds = r["references"], r["predictions"]
    if not refs:
        print(f"No samples evaluated for {display_name}, skipping metrics.")
        continue
    lang_wer, lang_cer = wer(refs, preds), cer(refs, preds)
    rows.append({
        "Language": display_name,
        "N": len(refs),
        "WER %": round(lang_wer * 100, 2),
        "CER %": round(lang_cer * 100, 2),
    })

summary_df = pd.DataFrame(rows).set_index("Language")
print(summary_df.to_string())

out_csv = "/kaggle/working/indicwav2vec_kathbath_summary.csv"
summary_df.to_csv(out_csv)
print(f"\nSaved summary to {out_csv}")


             N  WER %  CER %
Language                    
Bengali   1783  22.69   4.68

Saved summary to /kaggle/working/indicwav2vec_kathbath_summary.csv


## Notes

- **Path issues:** if you get a `manifest.json not found` error, print
  `os.listdir(KATHBATH_ROOT)` to see the exact top-level folder name Kaggle mounted your
  dataset under, and adjust `KATHBATH_ROOT` accordingly (Kaggle sometimes nests an extra
  folder level depending on how the dataset was zipped/uploaded).
- **`audio_filepath` resolution:** manifests can have absolute paths from the original
  download machine, paths already prefixed with `kathbath/bengali/wavs/...`, or bare
  filenames - all of which break naive joining against `KATHBATH_ROOT`. The notebook
  scans the language folder once (`build_audio_index`) and matches every manifest entry
  by filename, so it's robust regardless of subfolder naming (`wav` vs `wavs`) or path
  prefixes in the manifest - no manifest editing needed.
- **Resuming across sessions:** `checkpoint_bengali.json` in `/kaggle/working` holds
  every reference/prediction gathered so far. If a session times out, save
  `/kaggle/working` as a Kaggle Dataset, start a new session, copy the checkpoint file
  back into `/kaggle/working`, and re-run - it resumes from `last_index + 1` automatically.
- **One decode pass, not two:** IndicWav2Vec is a single fine-tuned wav2vec 2.0 + CTC
  checkpoint per language (via the HF `automatic-speech-recognition` pipeline), so there's
  no CTC/RNNT split to report here - just one WER/CER column, unlike the IndicConformer
  notebook's CTC and RNNT columns.
- **Extending to more languages:** IndicWav2Vec has a separate HF repo per language
  (`ai4bharat/indicwav2vec_v1_<lang>`). To add another language, extract it in the
  download/extract cell, load a second `pipeline(...)` pointed at that language's repo,
  and add a matching entry to `LANGUAGES` (with a small tweak to `evaluate_language` to
  pick the right `pipe` per language, since - unlike IndicConformer - there's no single
  multilingual model shared across languages here).
